# MDTF Variable Translation: Reusable Template

> Disclaimer: This notebook is a template for new POD notebooks and should be adapted to a specific variable or convention pair before being used as a final example. This notebook was created with GitHub Copilot assistance and is still being vetted.

This notebook shows the minimal workflow for translating variable names between MDTF conventions. It is intentionally small and reusable: it focuses on the framework API, the fieldlist lookup, and the translation check, without depending on a specific scientific demo.

For more information on variable translation, see: [variable_translation_developer_guide.ipynb](variable_translation_developer_guide.ipynb)

## 1. Notebook setup

Use this cell to locate the MDTF checkout and make imports work from either the repo root or a subdirectory.

In [10]:
import sys
from pathlib import Path

candidate_dirs = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in candidate_dirs if (path / "src" / "translation.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start Jupyter in the MDTF checkout or one of its subdirectories.")

sys.path.insert(0, str(repo_root))
print(f"MDTF repo root: {repo_root}")

MDTF repo root: /work5/j3k/mdtf_main/MDTF-diagnostics


## 2. Configure the example

Update the values below for a new variable pair or a new POD example.

In [11]:
SOURCE_CONVENTION = "CMIP"
TARGET_CONVENTION = "GFDL"
REQUESTED_VAR = "tas"

# Recommended for future reuse:
# - change these values for each new example
# - keep the rest of the notebook unchanged
#
# Example pair:
#   CMIP -> GFDL, tas
#
# The source variable name is determined dynamically from the translator
# by matching metadata between the requested POD variable and the target
# convention's fieldlist entries.

print(f"Translate from {SOURCE_CONVENTION} {REQUESTED_VAR} to {TARGET_CONVENTION}")

Translate from CMIP tas to GFDL


## 3. Load fieldlists

This mirrors the framework setup path: initialize the singleton translator and read all convention tables.

In [12]:
from src.translation import VariableTranslator

translator = VariableTranslator(str(repo_root))
translator.read_conventions(str(repo_root))

print("Loaded conventions:", sorted(translator.conventions.keys()))

Loaded conventions: ['cesm', 'cmip', 'gfdl']


## 4. Validate the mapping

Before translating a full varlist entry, confirm that the source and target conventions share the expected metadata and that the mapping is meaningful.

In [13]:
requested_field = translator.get_convention(SOURCE_CONVENTION).lut[REQUESTED_VAR]

target_fieldlist = translator.get_convention(TARGET_CONVENTION)
matched_entries = [
    (name, entry)
    for name, entry in target_fieldlist.lut.items()
    if entry.get("standard_name") == requested_field.get("standard_name")
    and entry.get("realm") == requested_field.get("realm")
    and entry.get("modifier") == requested_field.get("modifier")
    and entry.get("units") == requested_field.get("units")
]
if not matched_entries:
    raise ValueError(
        f"No matching field in {TARGET_CONVENTION} for {REQUESTED_VAR} "
        f"using metadata {requested_field}"
    )
if len(matched_entries) > 1:
    raise ValueError(
        f"Multiple matches in {TARGET_CONVENTION} for {REQUESTED_VAR}: "
        f"{[name for name, _ in matched_entries]}"
    )

# This is the resolved variable name in the target convention after matching by
# metadata. For a new notebook, this is what you will typically use when you
# want to rename or reference the source variable in the POD data.
RESOLVED_SOURCE_VAR = matched_entries[0][0]
source_field = target_fieldlist.lut[RESOLVED_SOURCE_VAR]


print("Requested field:", requested_field)
print("Field metadata matches as expected.")

print("Resolved source variable:", RESOLVED_SOURCE_VAR)
assert requested_field["modifier"] == source_field["modifier"]

print("Target field:", source_field)
assert requested_field["realm"] == source_field["realm"]

assert requested_field["standard_name"] == source_field["standard_name"]

Requested field: OrderedDict({'standard_name': 'air_temperature', 'realm': 'atmos', 'units': 'K', 'ndim': 3, 'modifier': 'atmos_height', 'long_name': 'air temperature', 'alternate_standard_names': [], 'name': 'tas'})
Field metadata matches as expected.
Resolved source variable: t_ref
Target field: OrderedDict({'standard_name': 'air_temperature', 'realm': 'atmos', 'long_name': 'temperature at 2 m', 'units': 'K', 'ndim': 3, 'modifier': 'atmos_height', 'alternate_standard_names': []})


## 5. Minimal reusable translation example

This is the core template for a POD notebook: build a minimal POD-like object, ask the framework to translate the requested variable, and confirm the result.

There are two equivalent ways to perform the same translation in this example:

- `translated_var` calls the target convention's fieldlist directly. This mirrors the runtime path after the framework has already chosen the data fieldlist.
- `translated_via_wrapper` calls `VariableTranslator.translate(...)`, which selects the fieldlist for you and then delegates to the same underlying method.

The difference is in the dispatch step, not in the final variable mapping. Both should resolve to the same target variable name.

In [14]:
from types import SimpleNamespace
from src.varlist_util import Varlist

example_settings = {
    "data": {"frequency": "day", "realm": "atmos"},
    "dimensions": {
        "lat": {
            "axis": "Y",
            "standard_name": "latitude",
            "units": "degrees_north",
        },
        "lon": {
            "axis": "X",
            "standard_name": "longitude",
            "units": "degrees_east",
        },
        "time": {"axis": "T", "standard_name": "time", "units": "days"},
    },
    "varlist": {
        REQUESTED_VAR: {
            "standard_name": requested_field["standard_name"],
            "long_name": requested_field.get("long_name", ""),
            "units": requested_field["units"],
            "modifier": requested_field.get("modifier", ""),
            "dimensions": ["time", "lat", "lon"],
        }
    },
}

pod = SimpleNamespace(
    pod_data=example_settings["data"],
    pod_dims=example_settings["dimensions"],
    pod_vars=example_settings["varlist"],
    pod_settings={"convention": SOURCE_CONVENTION},
    _log_name="translation_template",
)

requested_var = Varlist.from_struct(pod).vars[0]

# Direct fieldlist call: this mirrors the runtime path after the framework has
# already selected the target convention's fieldlist.
fieldlist = translator.get_convention(TARGET_CONVENTION)
translated_var = fieldlist.translate(requested_var, SOURCE_CONVENTION)

# Wrapper call: this is the higher-level API. The translator chooses the
# fieldlist internally and then calls the same underlying translate() method.
translated_via_wrapper = translator.translate(
    requested_var,
    to_convention=TARGET_CONVENTION,
    from_convention=SOURCE_CONVENTION,
)

print("Requested:", requested_var.name, requested_var.standard_name)
print("Translated:", translated_var.name, translated_var.standard_name)
assert translated_var.name == RESOLVED_SOURCE_VAR
assert translated_via_wrapper.name == translated_var.name

Requested: tas air_temperature
Translated: t_ref air_temperature


## 6. Reuse checklist for future POD notebooks

Before integrating this functionality into your own notebook, update the following:
- variable pair and conventions
- required dimensions and metadata
- minimal assertions
- any dataset paths or case metadata